# 11. Models from decks

A deck is a plain-text table describing a spherically symmetric model:
one row per knot, with the radius first and then one column per
field, the rows ordered by radius. A radius that appears twice marks a
boundary: the first of the two rows belongs to the layer below it and
the second to the layer above. Decks are how PREM and the models used
with the mineos normal-mode code are published.

What the columns are called and what the header lines mean is a
`DeckFormat`; `MINEOS` is the format of mineos and PREM decks, and
`GIA` that of a viscoelastic model with a viscosity column.
`read_deck` turns a file into a `Deck`, which is the numbers and the
header and nothing else. `deck_layers` interpolates every column,
layer by layer, with a piecewise polynomial through the knots, so the
base fields are polynomials and the arithmetic of the earlier
tutorials is exact on them; the moduli are products of them. The
mixins then add everything else in the usual way.

This tutorial reads PREM's own 200-knot deck as a `MineosModel` and
compares it with the polynomial `PREM`. It then writes a deck in a
format of its own, PREM's elastic part with a viscosity column on the
solid layers, and reads it back as a model type of its own, of the
kind used for glacial isostatic adjustment.

This tutorial plots, so it needs the `plot` extra (matplotlib). The
figure is written to `examples/figures/`.

In [ ]:
# Generated from examples/tutorials/11_models_from_decks.py.
# Edit the script, not this notebook.  On Google Colab the library is
# installed first, from GitHub, with the extras this tutorial needs.
# Everywhere, the notebook is given the script's place, so that the files
# it writes and the data it reads are found as the script finds them; its
# figures are shown here rather than saved.
import subprocess
import sys
from pathlib import Path
import urllib.request

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "planetmodel[plot] @ "
         "git+https://github.com/da380/planetmodel@main"],
        check=True)
here = Path.cwd()
tutorials = here.parent if here.name == "notebooks" else here / "examples" / "tutorials"
__file__ = str(tutorials / "11_models_from_decks.py")
sys.argv = [__file__]
(tutorials.parent / "figures").mkdir(parents=True, exist_ok=True)
if "get_ipython" in globals():          # figures shown in the notebook, sharp
    get_ipython().run_line_magic("config", "InlineBackend.figure_format = 'retina'")

for name in ('prem.200', 'prem.nocrust'):
    target = tutorials.parent / "data" / name
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/da380/planetmodel/main/examples/data/"
            + name, target)

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

from planetmodel import (GIA, PREM, Deck, Elastic, Geometry, MINEOS, MineosModel,
                         Model, SelfGravitating, Tabulated, Viscoelastic, deck_layers,
                         gravity, read_deck, testing, write_deck)
from planetmodel.deck import deck_knots

DATA = Path(__file__).resolve().parent.parent / "data"
FIGURES = Path(__file__).resolve().parent.parent / "figures"
FIGURES.mkdir(exist_ok=True)

## A mineos deck

`read_deck` with the `MINEOS` format reads three header lines, a
title, then `ifanis tref ifdeck`, then `nknot nic noc`, and nine
columns in SI units: the radius, the density, the vertical and
horizontal P and S velocities, the two quality factors and the
anisotropy parameter eta. The layering comes from the repeated radii,
thirteen layers here. The header's `nic` and `noc` say which knots end
the inner and the outer core. `layers` returns a slice of the knots
for each layer; the outer core's S velocity is zero on every knot, as
a fluid's must be.

In [ ]:
deck = read_deck(DATA / "prem.200", MINEOS)
print(deck)
print("header:", dict(deck.header))
print("boundaries (km):", np.round(deck.boundaries / 1e3, 1))
s = deck.layers()[1]
print("outer core knots:", s, "| vsv there:", np.unique(deck["vsv"][s]))

## The model of a deck

`MineosModel` is the model type of that format. The columns become the
base fields by a cubic spline through each layer's knots. `Elastic`
adds the five moduli beside them, the header names the two core layers
and their boundaries and sets the reference frequency of the
attenuation model, and the model can do everything a `PREM` can. It
keeps its deck, its knots and its header, and `to_deck` writes it
back out.

In [ ]:
model = MineosModel(deck)
print(model)
print("layers:", [layer.name for layer in model.layers][:3], "...",
      "| interfaces:", [f.name for f in model.geometry.interfaces][:2])
print("the outer core holds:", model.layer("outer_core").names)
print("reference period:", 2 * np.pi / model.reference_omega(), "s")
testing.check_model(model)
print("check_model passes")

The deck is a 200-knot sampling of the polynomial PREM, and the spline
through it agrees with the polynomial to about a part in a million.
The moduli are products of splines and agree to the same order, and
so do the mass and the gravity that follows from the density.

In [ ]:
prem = PREM()
for x in (1000e3, 3000e3, 5000e3, 6300e3):
    i = model.skeleton.locate(x).layer
    d, p = model.layer(i), prem.layer(i)
    print(f"r = {x / 1e3:5.0f} km: rho {d['rho'](x):9.2f} vs {p['rho'](x):9.2f}   "
          f"A/A_prem - 1 = {d['A'](x) / p['A'](x) - 1:+.1e}")
print(f"mass: deck {model.mass():.7e}  prem {prem.mass():.7e}")
for x in (3480e3, 6371e3):
    print(f"g at {x / 1e3:4.0f} km: deck {gravity(model, x):.6f}  "
          f"prem {gravity(prem, x):.6f}")

## A deck of your own

A format is a set of column names and a description of the header.
The library ships one for glacial isostatic adjustment, `GIA`: seven
columns after the radius, PREM's elastic fields and a viscosity, and a
header of two lines, a title and the column names. A format of your
own is a `DeckFormat` given its column names, the number of header
lines and, if you want the header read into a mapping, the two
functions that read it and write it back.

The deck itself is built from PREM's elastic part, with no attenuation
and no ocean, sampled on eight knots per layer. The viscosity column
is set on the solid layers and NaN on the fluid outer core. A column
that is NaN on every knot of a layer is absent from that layer's
fields, which is how a rectangular table carries a field that some
layers do not have.

In [ ]:
elastic = prem.elastic().truncated(6368e3, name="surface")   # no Q, no ocean
viscosity = {"lower_mantle": 1e22, "upper_lower_mantle": 1e22,
             "lowermost_mantle": 1e22, "inner_core": 1e21}
radii, columns = [], {name: [] for name in GIA.columns}
for layer in elastic.layers:
    lo, hi = layer.interval
    r = np.linspace(lo, hi, 8)
    radii.append(r)
    for name in GIA.columns:
        if name == "viscosity":
            eta = np.nan if elastic.is_fluid(layer.index) else viscosity.get(layer.name,
                                                                            1e21)
            columns[name].append(np.full(r.size, eta))
        else:
            columns[name].append(layer[name](r))
gia_deck = Deck(np.concatenate(radii),
                {n: np.concatenate(v) for n, v in columns.items()},
                header={"title": "PREM, elastic, with a Maxwell viscosity"})
workdir = Path(tempfile.mkdtemp(prefix="planetmodel_"))
path = write_deck(workdir / "prem_gia.deck", gia_deck, GIA)
print(path.name, "written;", gia_deck)
print("\n".join(path.read_text().splitlines()[:4]))

The model type for that format is a short class. Its constructor
reads the deck, interpolates the layers, keeps the knots and the
header as the `Tabulated` mixin expects, and passes the geometry and
the layers to `Model`. The layer names are taken from the model the
deck was written from, since this format's header does not record
them. The elastic, gravity and viscoelastic mixins are the same as
before. A layer with a viscosity is a Maxwell body in shear, so
`frozen` gives the model with complex moduli at a chosen period. The
shear modulus of the lower mantle is its elastic value at ten years,
and has relaxed most of the way to zero at a hundred thousand years,
well past the Maxwell time of about two thousand years.

In [ ]:
class GIAModel(Elastic, SelfGravitating, Viscoelastic, Tabulated, Model):
    """PREM's elastic part with a Maxwell viscosity on the solid layers."""

    def __init__(self, source, *, kind="cubic"):
        d = read_deck(source, GIA)
        skeleton, layers = deck_layers(d, kind=kind)
        names = [lay.name for lay in elastic.layers]          # the same layering
        self.knots, self.header = deck_knots(d), d.header
        super().__init__(Geometry(skeleton, layer_names=names), layers)


gia = GIAModel(path)
print(gia, "|", gia.header["title"])
print("viscoelastic layers:",
      [lay.name for lay in gia.layers if gia.is_viscoelastic(lay.index)])
print("the outer core holds no viscosity:", "viscosity" not in gia.layer("outer_core"))
year = 3.15576e7
x = 4500e3
print(f"elastic L at {x / 1e3:.0f} km: "
      f"{gia.layer('lower_mantle')['L'](x) / 1e9:.2f} GPa")
for period in (10.0, 1e3, 1e5):
    at_period = gia.frozen(2 * np.pi / (period * year))
    L = at_period.layer("lower_mantle")["L"](x) / 1e9
    print(f"L at {period:8.0f} years: {L.real:+8.2f} {L.imag:+8.2f}i GPa   "
          f"({type(at_period).__name__})")
testing.check_model(gia)
print("check_model passes; the deck it now is:",
      gia.to_deck(columns=["rho", "viscosity"]))

## A picture

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed; no figure")
    raise SystemExit(0)

from planetmodel.plotting import radial_profile  # noqa: E402

fig, (left, middle, right) = plt.subplots(1, 3, figsize=(14, 6), sharey=True)
radial_profile(left, prem, "rho", scale=1e-3, value_scale=1e-3, color="0.6", lw=2.5,
               label="PREM, polynomial")
radial_profile(left, model, "rho", scale=1e-3, value_scale=1e-3, color="C3", lw=0.9,
               label="prem.200, spline")
for s in deck.layers():
    left.plot(deck["rho"][s] / 1e3, deck.radius[s] / 1e3, "k.", ms=2)
left.set_xlabel("rho (g/cm^3)"); left.set_ylabel("r (km)"); left.legend()
left.set_title("the deck's knots and its spline")
radial_profile(middle, model, "L", scale=1e-3, value_scale=1e-9, lw=1.2)
middle.set_xlabel("L (GPa)"); middle.set_title("a modulus, product of splines")
radial_profile(right, gia, "viscosity", scale=1e-3, lw=1.2)
right.set_xscale("log"); right.set_xlabel("viscosity (Pa s)")
right.set_title("the custom column, absent in the fluid")
fig.tight_layout()